In [4]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F


from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/data_hl19.csv"

# 0. CONFIG
MODEL_TYPE = "INFORMER" # Or "ANOMALY", "INFORMER"

# 1. Data Loading and Preprocessing
df = pd.read_csv(path)

# Store dates before dropping the column
dates = df['date'].copy()
df = df.drop(columns=['date', 'SGW BEARER SR'])

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df.values)
data_tensor = torch.tensor(data_scaled, dtype=torch.float32)

seq_len = 50
def create_sequences(data, seq_len):
    xs = []
    for i in range(len(data) - seq_len):
        x = data[i:(i + seq_len)]
        xs.append(x)
    return torch.stack(xs)

X = create_sequences(data_tensor, seq_len)
#X = X[:5000] # Using only the first 5000 sequences

dates_for_anomalies = dates[seq_len : seq_len + len(X)].reset_index(drop=True)


# 2. Model Definitions
class AnomalyAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        self.W_sigma = nn.Linear(d_model, 1)

    def forward(self, x):
        Q, K, V = x, x, x
        attn_output, attn_weights = self.attn(Q, K, V)

        sigma = torch.abs(self.W_sigma(x)) + 1e-6
        prior = torch.exp(-((torch.arange(x.size(1), device=x.device).unsqueeze(0) -
                             torch.arange(x.size(1), device=x.device).unsqueeze(1)) ** 2) / (2 * sigma.mean().item() ** 2))
        prior = prior.unsqueeze(0).repeat(x.size(0), 1, 1)
        prior = prior / (prior.sum(dim=-1, keepdim=True) + 1e-6)

        return attn_output, attn_weights, prior

class AnomalyTransformer(nn.Module):
    def __init__(self, d_model=18, n_heads=3, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)
            for _ in range(num_layers)
        ])
        self.attn = AnomalyAttention(d_model, n_heads)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        attn_output, attn_weights, prior = self.attn(x)
        out = self.fc_out(attn_output)
        return out, attn_weights, prior

# iTransformer Model
class iTransformer(nn.Module):
    def __init__(self, d_model=18, nhead=3, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.decoder = nn.Linear(d_model, d_model)

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

# Informer Autoencoder Model
class InformerAutoencoder(nn.Module):
    def __init__(self, d_model=18, nhead=3, num_encoder_layers=2, num_decoder_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

        # Using standard TransformerDecoderLayer for Informer's decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, src):
        # Encoder
        memory = self.encoder(src)

        # Decoder input (using src as target for reconstruction)
        tgt = src

        # Decoder
        output = self.decoder(tgt, memory)

        # Final linear layer
        out = self.fc_out(output)
        return out


# 3. Model Initialization and Training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if MODEL_TYPE == "ANOMALY":
    anomaly_d_model = 18
    anomaly_n_heads = 3
    if anomaly_d_model % anomaly_n_heads != 0:
      print(f"Warning: AnomalyTransformer d_model ({anomaly_d_model}) is not divisible by n_heads ({anomaly_n_heads}). Adjusting d_model to {anomaly_d_model + (anomaly_n_heads - anomaly_d_model % anomaly_n_heads)}")
      anomaly_d_model = anomaly_d_model + (anomaly_n_heads - anomaly_d_model % anomaly_n_heads)

    model = AnomalyTransformer(d_model=anomaly_d_model, n_heads=anomaly_n_heads).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    batch_size = 16
    print("Training Anomaly Transformer...")
    for epoch in range(10):
        total_loss = 0
        for i in range(0, X.size(0), batch_size):
            batch = X[i:i+batch_size].to(device)
            optimizer.zero_grad()
            outputs, attn, prior = model(batch)
            loss = criterion(outputs, batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.6f}")

    print("\nDetecting anomalies (Anomaly Transformer)...")
    with torch.no_grad():
        X_device = X.to(device)
        pred, attn, prior = model(X_device)

        attn_mean = attn.mean(dim=1)
        prior_mean = prior.mean(dim=1)

        discrepancy = torch.abs(attn_mean - prior_mean)
        anomaly_score = discrepancy.mean(dim=-1).cpu()

        # Calculate MSE and MAE
        mse = F.mse_loss(pred.cpu(), X_device.cpu())
        mae = F.l1_loss(pred.cpu(), X_device.cpu())
        print(f"Anomaly Transformer - MSE: {mse.item():.6f}, MAE: {mae.item():.6f}")


    print("Anomaly score shape:", anomaly_score.shape)
    print("Top 50 potential anomalies:", anomaly_score.topk(50))

    top_k_values, top_k_indices = anomaly_score.topk(50)
    print("\nTop 50 Anomalies with Dates:")
    # Convert top_k_indices to a list of integers before indexing
    for i in range(50):
        date = dates_for_anomalies.iloc[top_k_indices[i].item()]
        score = top_k_values[i]
        print(f"Date: {date}, Anomaly Score: {score:.6f}")


elif MODEL_TYPE == "ITRANSFORMER":
    itransformer_d_model = 18
    itransformer_n_heads = 3
    if itransformer_d_model % itransformer_n_heads != 0:
      print(f"Warning: iTransformer d_model ({itransformer_d_model}) is not divisible by n_heads ({itransformer_n_heads}). Adjusting d_model to {itransformer_d_model + (itransformer_n_heads - itransformer_d_model % itransformer_n_heads)}")
      itransformer_d_model = itransformer_d_model + (itransformer_n_heads - itransformer_d_model % itransformer_n_heads)

    model = iTransformer(d_model=itransformer_d_model, nhead=itransformer_n_heads, num_layers=3).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    class DummyDataset(Dataset):
        def __init__(self, data):
            self.data = data
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            return self.data[idx], self.data[idx]

    dummy_dataset = DummyDataset(X)
    loader = DataLoader(dummy_dataset, batch_size=16, shuffle=True)

    print("Training iTransformer...")
    epochs = 10
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in loader:
            x, y = [b.to(device) for b in batch]
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.6f}")

    print("\nDetecting anomalies (iTransformer)...")
    model.eval()
    reconstruction_errors = []
    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in loader:
            x, y = [b.to(device) for b in batch]
            output = model(x)
            errors = torch.mean((output - y)**2, dim=[1, 2]).cpu().numpy()
            reconstruction_errors.extend(errors)
            predictions.append(output.cpu())
            actuals.append(y.cpu())

    reconstruction_errors = np.array(reconstruction_errors)
    predictions = torch.cat(predictions)
    actuals = torch.cat(actuals)

    # Calculate MSE and MAE
    mse = F.mse_loss(predictions, actuals)
    mae = F.l1_loss(predictions, actuals)
    print(f"iTransformer - MSE: {mse.item():.6f}, MAE: {mae.item():.6f}")


    reconstruction_errors_tensor = torch.tensor(reconstruction_errors)
    top_k = 50
    top_k_values, top_k_indices = torch.topk(reconstruction_errors_tensor, top_k)

    print(f"\nTop {top_k} Anomalies with Dates (iTransformer):")
    # Convert top_k_indices to a list of integers before indexing
    for i in range(top_k):
        date = dates_for_anomalies.iloc[top_k_indices[i].item()]
        score = top_k_values[i]
        print(f"Date: {date}, Anomaly Score: {score:.6f}")


elif MODEL_TYPE == "INFORMER":
    informer_d_model = 18
    informer_n_heads = 3 # Changed to 3 to be divisible by 18
    if informer_d_model % informer_n_heads != 0:
        print(f"Warning: Informer d_model ({informer_d_model}) is not divisible by n_heads ({informer_n_heads}). Adjusting d_model to {informer_d_model + (informer_n_heads - informer_d_model % informer_n_heads)}")
        informer_d_model = informer_d_model + (informer_n_heads - informer_d_model % informer_n_heads)


    # Using standard Transformer Autoencoder (Encoder-Decoder)
    model = InformerAutoencoder(
        d_model=informer_d_model,
        nhead=informer_n_heads,
        num_encoder_layers=2,
        num_decoder_layers=2
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Dummy DataLoader for Informer - In a real case, replace with your data
    class DummyInformerDataset(Dataset):
        def __init__(self, data):
            self.data = data
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            # For reconstruction, input and target are the same
            return self.data[idx], self.data[idx]

    informer_dataset = DummyInformerDataset(X)
    informer_loader = DataLoader(informer_dataset, batch_size=16, shuffle=True)


    print("Training Informer (Transformer Autoencoder)...")
    epochs = 10
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in informer_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(informer_loader):.6f}")

    print("\nDetecting anomalies (Informer)...")
    model.eval()
    reconstruction_errors = []
    predictions = []
    actuals = []
    with torch.no_grad():
        for x, y in informer_loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            errors = torch.mean((output - y)**2, dim=[1, 2]).cpu().numpy()
            reconstruction_errors.extend(errors)
            predictions.append(output.cpu())
            actuals.append(y.cpu())

    predictions = torch.cat(predictions)
    actuals = torch.cat(actuals)
    reconstruction_errors = np.array(reconstruction_errors)

    # Calculate MSE and MAE
    mse = F.mse_loss(predictions, actuals)
    mae = F.l1_loss(predictions, actuals)
    print(f"Informer - MSE: {mse.item():.6f}, MAE: {mae.item():.6f}")

    reconstruction_errors_tensor = torch.tensor(reconstruction_errors)
    top_k = 50
    top_k_values, top_k_indices = torch.topk(reconstruction_errors_tensor, top_k)

    print(f"\nTop {top_k} Anomalies with Dates (Informer):")
    # Convert top_k_indices to a list of integers before indexing
    for i in range(top_k):
        date = dates_for_anomalies.iloc[top_k_indices[i].item()]
        score = top_k_values[i]
        print(f"Date: {date}, Anomaly Score: {score:.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


Training Informer (Transformer Autoencoder)...
Epoch 1/10 - Loss: 0.010336
Epoch 2/10 - Loss: 0.001753
Epoch 3/10 - Loss: 0.001042
Epoch 4/10 - Loss: 0.000703
Epoch 5/10 - Loss: 0.000485
Epoch 6/10 - Loss: 0.000359
Epoch 7/10 - Loss: 0.000281
Epoch 8/10 - Loss: 0.000232
Epoch 9/10 - Loss: 0.000197
Epoch 10/10 - Loss: 0.000171

Detecting anomalies (Informer)...
Informer - MSE: 0.000176, MAE: 0.009550

Top 50 Anomalies with Dates (Informer):
Date: 2/18/2025 13:10, Anomaly Score: 0.001085
Date: 2/26/2025 19:10, Anomaly Score: 0.001083
Date: 3/20/2025 19:55, Anomaly Score: 0.001083
Date: 2/22/2025 6:35, Anomaly Score: 0.001082
Date: 2/28/2025 15:10, Anomaly Score: 0.001082
Date: 3/5/2025 10:45, Anomaly Score: 0.001082
Date: 3/18/2025 11:15, Anomaly Score: 0.001081
Date: 2/13/2025 10:15, Anomaly Score: 0.001081
Date: 3/24/2025 11:15, Anomaly Score: 0.001081
Date: 2/13/2025 12:10, Anomaly Score: 0.001081
Date: 3/23/2025 19:45, Anomaly Score: 0.001080
Date: 2/1/2025 7:15, Anomaly Score: 0.001